# 06. Results Summary: OOS Detection

Итоговые результаты экспериментов по Out-of-Scope детекции
на датасете CLINC150.

## Содержание
1. Setup
2. Обзор результатов
3. Сравнение по источникам (standard vs deeppavlov)
4. Few-shot Results
5. Scaling Curve
6. Анализ конфигурации AutoIntent
7. Проверка гипотез
8. Выводы и рекомендации

## 1. Setup

In [ ]:
# Load environment variables from .env (macOS ARM fix)
from dotenv import load_dotenv
load_dotenv("../../../.env")  # project root

import sys
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project paths
NOTEBOOK_DIR = Path.cwd()
TASK_DIR = NOTEBOOK_DIR.parent
PROJECT_ROOT = TASK_DIR.parent.parent
PROCESSED = TASK_DIR / "data" / "processed"
RESULTS = TASK_DIR / "results"
RUNS = TASK_DIR / "runs"

sys.path.insert(0, str(PROJECT_ROOT))

# Load all results
metrics_all = json.loads((RESULTS / "metrics.json").read_text())
df = pd.DataFrame(metrics_all)

# Extract source from extra field
df['source'] = df['extra'].apply(lambda x: x.get('source', 'unknown') if isinstance(x, dict) else 'unknown')
df['embedder_fixed'] = df['extra'].apply(lambda x: x.get('embedder_fixed', True) if isinstance(x, dict) else True)

print(f"Total entries: {len(df)}")
print(f"Sources: {df['source'].unique().tolist()}")
print(f"Models: {df['model_name'].nunique()}")

## 2. Обзор результатов

### 2.1 Общая статистика

### 2.2 Записей по source и model

In [ ]:
# Summary by source
print("=== Записей по source ===")
print(df.groupby('source').size())
print()

# Models for analysis (exclude autoembedder since it's identical to fixed)
# Note: AutoML chose the same embedder, so results are identical
models_order = [
    "autointent_classic-light",
    "cosine_e5large_threshold",
    "cosine_minilm_threshold", 
    "tfidf_threshold",
]

print("=== F1 OOS (mean ± std) по моделям и source ===")
summary = df[df['model_name'].isin(models_order)].groupby(['model_name', 'source'])['f1_oos'].agg(['mean', 'std']).round(4)
display(summary.unstack('source'))

## 3. Сравнение по источникам (standard vs deeppavlov)

### Почему некоторые результаты идентичны?

**Cosine бейзлайны одинаковые для standard и deeppavlov** — это ожидаемо:
- `fit()` использует только in-scope примеры (идентичны в обоих источниках)
- Validation и test sets идентичны
- Единственное различие — количество OOS в train (100 vs 200), но cosine модель их не использует

**TF-IDF отличается** — LogReg видит OOS при обучении

**AutoIntent fixed vs autoembedder идентичны** — AutoML выбрал тот же embedder (`e5-large-instruct`)

In [ ]:
# Full train results by source
df_full = df[(df["mode"] == "full") & (df["model_name"].isin(models_order))].copy()

print("=== Full Train Results by Source ===\n")

for source in ['deeppavlov', 'standard']:
    print(f"--- {source.upper()} ---")
    source_df = df_full[df_full['source'] == source].copy()
    source_df["sort_key"] = source_df["model_name"].apply(lambda x: models_order.index(x))
    source_df = source_df.sort_values("sort_key")
    
    cols = ["model_name", "oos_recall", "in_domain_acc", "f1_oos", "auroc", "latency_ms"]
    table = source_df[cols].copy()
    table.columns = ["Model", "OOS Recall", "In-D Acc", "F1 OOS", "AUROC", "Lat(ms)"]
    
    # Format
    for col in ["OOS Recall", "In-D Acc", "F1 OOS", "AUROC"]:
        table[col] = table[col].apply(lambda x: f"{x:.3f}")
    table["Lat(ms)"] = table["Lat(ms)"].apply(lambda x: f"{x:.2f}")
    
    display(table)
    print()

## 4. Few-shot Results (deeppavlov, сравнимо с Table 3)

Для сравнения с AutoIntent Table 3 используем **deeppavlov** source (200 OOS в train).

In [ ]:
# Few-shot results for deeppavlov (comparable to Table 3)
df_dp = df[df['source'] == 'deeppavlov']
n_shots_values = [10, 20, 50]

summary_rows = []
for model in models_order:
    row = {"Model": model}
    for n in n_shots_values:
        subset = df_dp[(df_dp["model_name"] == model) & (df_dp["n_shots"] == n)]
        if len(subset) > 0:
            mean = subset["f1_oos"].mean()
            std = subset["f1_oos"].std() if len(subset) > 1 else 0
            row[f"{n}-shot"] = f"{mean:.3f} ± {std:.3f}" if std > 0 else f"{mean:.3f}"
        else:
            row[f"{n}-shot"] = "—"
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

print("F1 OOS by model and n_shots (deeppavlov, mean ± std across 3 seeds)")
print("=" * 70)
display(summary_df)

## 5. Scaling Curve

In [ ]:
# Models for scaling curve
scaling_models = [
    "autointent_classic-light",
    "cosine_e5large_threshold",
    "cosine_minilm_threshold",
]

# Use deeppavlov for Table 3 comparison
df_dp = df[df['source'] == 'deeppavlov']

# Collect data for all modes
scaling_data = []

for model in scaling_models:
    model_df = df_dp[df_dp["model_name"] == model]
    
    # Few-shot: mean by n_shots
    for n in [10, 20, 50]:
        subset = model_df[model_df["n_shots"] == n]
        if len(subset) > 0:
            scaling_data.append({
                "model": model,
                "mode": f"{n}-shot",
                "f1_oos_mean": subset["f1_oos"].mean(),
                "f1_oos_std": subset["f1_oos"].std() if len(subset) > 1 else 0,
            })
    
    # Full train
    full_subset = model_df[model_df["mode"] == "full"]
    if len(full_subset) > 0:
        scaling_data.append({
            "model": model,
            "mode": "full train",
            "f1_oos_mean": full_subset["f1_oos"].values[0],
            "f1_oos_std": 0,
        })

scaling_df = pd.DataFrame(scaling_data)
print(f"Scaling data points: {len(scaling_df)} (deeppavlov source)")

In [ ]:
# Plot scaling curve
fig, ax = plt.subplots(figsize=(10, 6))

# X-axis labels (categorical)
x_labels = ["10-shot", "20-shot", "50-shot", "full train"]
x_pos = np.arange(len(x_labels))

# Colors and markers
styles = {
    "autointent_classic-light": {"color": "#2ecc71", "marker": "^", "label": "AutoIntent"},
    "cosine_e5large_threshold": {"color": "#9b59b6", "marker": "s", "label": "E5-Large + threshold"},
    "cosine_minilm_threshold": {"color": "#3498db", "marker": "o", "label": "MiniLM + threshold"},
}

# Plot each model
for model in scaling_models:
    model_data = scaling_df[scaling_df["model"] == model]
    
    # Get values in correct order
    y_vals = []
    y_errs = []
    for mode in x_labels:
        row = model_data[model_data["mode"] == mode]
        if len(row) > 0:
            y_vals.append(row["f1_oos_mean"].values[0])
            y_errs.append(row["f1_oos_std"].values[0])
        else:
            y_vals.append(np.nan)
            y_errs.append(0)
    
    style = styles[model]
    ax.errorbar(
        x_pos, y_vals, yerr=y_errs,
        marker=style["marker"], markersize=10,
        color=style["color"], linewidth=2, capsize=5,
        label=style["label"],
    )

# Reference line: AutoIntent Table 3
REFERENCE_F1 = 0.768
ax.axhline(y=REFERENCE_F1, color="red", linestyle="--", linewidth=2)
ax.annotate(
    "AutoIntent Table 3†",
    xy=(3.2, REFERENCE_F1), fontsize=9, color="red",
    verticalalignment="bottom",
)

ax.set_xticks(x_pos)
ax.set_xticklabels(x_labels)
ax.set_xlabel("Training data", fontsize=12)
ax.set_ylabel("F1 OOS", fontsize=12)
ax.set_title("Scaling Curve: F1 OOS vs Training Data Size (deeppavlov)", fontsize=14)
ax.set_ylim(0.2, 0.9)
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS / "summary_scaling_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {RESULTS / 'summary_scaling_curve.png'}")

## 6. Анализ конфигурации AutoIntent

Проверим, какие embedder и модули AutoML выбирает в режиме `--no-fix-embedder`.

In [ ]:
# Analyze AutoIntent configurations from saved models
import yaml
from collections import Counter

autoembedder_dirs = list(RUNS.glob("autointent_classic-light_autoembedder_*"))
print(f"Found {len(autoembedder_dirs)} autoembedder model directories\n")

embedders = []
modules = []
thresholds = []

for model_dir in sorted(autoembedder_dirs):
    config_file = model_dir / "inference_config.yaml"
    if config_file.exists():
        config = yaml.safe_load(config_file.read_text())
        
        # Extract embedder
        for node in config:
            if node.get('node_type') == 'scoring':
                emb = node.get('module_config', {}).get('embedder_config', {}).get('model_name', 'unknown')
                mod = node.get('module_name', 'unknown')
                embedders.append(emb)
                modules.append(mod)
            elif node.get('node_type') == 'decision':
                thresh = node.get('module_config', {}).get('thresh', None)
                if thresh:
                    thresholds.append(thresh)

print("=== Embedder выбранный AutoML ===")
for emb, count in Counter(embedders).items():
    print(f"  {emb}: {count}/{len(autoembedder_dirs)} runs")

print("\n=== Scoring module ===")
for mod, count in Counter(modules).items():
    print(f"  {mod}: {count}/{len(autoembedder_dirs)} runs")

print(f"\n=== Threshold statistics ===")
print(f"  Mean: {np.mean(thresholds):.4f}")
print(f"  Std:  {np.std(thresholds):.4f}")
print(f"  Min:  {np.min(thresholds):.4f}")
print(f"  Max:  {np.max(thresholds):.4f}")

In [ ]:
# Compare fixed vs autoembedder thresholds
fixed_dirs = list(RUNS.glob("autointent_classic-light_standard_*")) + list(RUNS.glob("autointent_classic-light_deeppavlov_*"))
print(f"Fixed embedder models: {len(fixed_dirs)}")

fixed_thresholds = []
for model_dir in sorted(fixed_dirs):
    config_file = model_dir / "inference_config.yaml"
    if config_file.exists():
        config = yaml.safe_load(config_file.read_text())
        for node in config:
            if node.get('node_type') == 'decision':
                thresh = node.get('module_config', {}).get('thresh', None)
                if thresh:
                    fixed_thresholds.append(thresh)

print(f"\n=== Сравнение threshold (fixed vs autoembedder) ===")
print(f"Fixed embedder:  mean={np.mean(fixed_thresholds):.4f}, std={np.std(fixed_thresholds):.4f}")
print(f"Autoembedder:    mean={np.mean(thresholds):.4f}, std={np.std(thresholds):.4f}")

# Are they identical?
if len(fixed_thresholds) == len(thresholds):
    identical = all(abs(f - a) < 1e-6 for f, a in zip(sorted(fixed_thresholds), sorted(thresholds)))
    print(f"\nThresholds identical: {identical}")

### Выводы по конфигурации AutoML

1. **Embedder:** AutoML стабильно выбирает `intfloat/multilingual-e5-large-instruct` во всех 20 прогонах (оба source, все n_shots, все seeds)
2. **Scoring module:** Всегда `linear` (линейный классификатор поверх embeddings)
3. **Decision module:** Всегда `threshold` с калиброванным порогом

**Вывод:** Режим `--no-fix-embedder` не даёт преимуществ для CLINC150 — AutoML выбирает тот же embedder, что мы фиксируем по умолчанию. Можно всегда использовать фиксированный embedder для экономии времени на AutoML-оптимизацию.

## 7. Проверка гипотез

| # | Гипотеза | Метод | Ключевой результат | Вывод |
|---|----------|-------|-------------------|-------|
| HYP-002 | Асимметричная cost function | Постфактум threshold на val, α:β ∈ {1:1, 2:1, 5:1, 10:1}, 20-shot, 3 seeds | Recall ↑ до 0.976, но FPR ↑ до 0.247; нестабильно при 2:1 (std Acc=0.045) | ⚠️ Технически работает, trade-off невыгоден для стандартного guardrail |

Проверенная гипотеза показала, что стандартная F1-оптимизация AutoIntent уже обеспечивает хороший guardrail-баланс. Асимметричная cost function остаётся configurable knob для сценариев с жёсткими требованиями к Recall, но не является улучшением пайплайна по умолчанию.

HYP-001 (per-intent threshold calibration) отложена: в few-shot режиме недостаточно данных на кластер для надёжной калибровки.

## 8. Выводы и рекомендации

### Главные результаты

**AutoIntent classic-light применим как OOS-guardrail.** В few-shot режиме (20-shot) на deeppavlov достигает F1 OOS ≈ 0.80, превосходя cosine-бейзлайны. AutoML-оптимизация стабильно выбирает linear + threshold и embedder `e5-large-instruct`.

### Ограничения

1. **Cosine бейзлайны не зависят от source** — используют только in-scope примеры, которые идентичны
2. **AutoML выбирает тот же embedder** — autointent_classic-light и autoembedder дают идентичные результаты
3. **TF-IDF — единственный бейзлайн, зависящий от source** — LogReg видит OOS при обучении

### Рекомендации для production

- Использовать **deeppavlov** source для сравнения с AutoIntent Table 3
- Минимальный порог данных — 20 примеров на интент
- AutoML embedder optimization не требуется — e5-large-instruct оптимален
- Использовать фиксированный embedder для экономии времени

### Дальнейшие шаги

- Jailbreak Detection
- Сравнение в протоколе TEXTOIR для ADB/DA-ADB/DETER